# BossAssistant — Multi-Department RAG with Knowledge Graph + Routing

**Assignment:** Build a `BossAssistant` that routes HR and Finance questions to department-specific RAG pipelines.


## 1. Environment Setup

In [1]:
!pip install --quiet \
    langchain \
    langchain-community \
    langchain-nvidia-ai-endpoints \
    langchain-huggingface \
    langchain-neo4j \
    neo4j \
    tiktoken \
    sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.0/234.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 10.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sou

### API Keys

You need these in Colab `userdata`:

| Secret | Where to get it |
|---|---|
| `NVIDIA_API_KEY` | https://build.nvidia.com → generate `nvapi-...` key |
| `NEO4J_URI` | https://neo4j.com/cloud/aura-free → create instance → copy URI |
| `NEO4J_USERNAME` | Neo4j Aura (usually `neo4j`) |
| `NEO4J_PASSWORD` | Neo4j Aura (shown once — save it!) |
| `LANGCHAIN_API_KEY` | https://smith.langchain.com → Settings |

In [2]:
import os
from google.colab import userdata

os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")

os.environ["NEO4J_URI"] = userdata.get("NEO4J_URI")
os.environ["NEO4J_USERNAME"] = userdata.get("NEO4J_USERNAME")
os.environ["NEO4J_PASSWORD"] = userdata.get("NEO4J_PASSWORD")
os.environ["NEO4J_DATABASE"] = userdata.get("NEO4J_DATABASE")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = userdata.get("LANGCHAIN_PROJECT")

print("Environment ready.")

Environment ready.


## 2. Initialize Models
in this RAG pipeline, i will use qwen/qwen3-next-80b-a3b-instruct for text-to-text generation, nvidia/nv-embedqa-e5-v5 for the embedding

In [3]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings

llm = ChatNVIDIA(
    model="qwen/qwen3-next-80b-a3b-instruct",
    temperature=0,
    max_tokens=1024,
)

embedder = NVIDIAEmbeddings(model="nvidia/nv-embedqa-e5-v5", truncate="END")

# Smoke test both
print("LLM:", llm.invoke("Reply with exactly: OK").content)
print(f"Embedding dim: {len(embedder.embed_query('test'))}")

/tmp/ipykernel_9927/122905481.py:3: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  llm = ChatNVIDIA(


LLM: OK
Embedding dim: 1024


## 3. Synthetic Department Documents (8 each)

In [4]:
from langchain_core.documents import Document

hr_docs = [
    Document(
        page_content="""Paid Time Off (PTO) Policy. All full-time employees accrue 15 days of paid vacation per year during their first three years of employment. After three years, accrual increases to 20 days per year. PTO is accrued monthly at 1.25 days per month (or 1.67 days for tenured employees). Unused PTO can be carried over up to a maximum of 5 days into the next calendar year. Any excess is forfeited on January 1st. PTO requests must be submitted through Workday at least two weeks in advance for requests longer than 3 consecutive days.""",
        metadata={"source": "HR-001", "title": "PTO Policy", "department": "hr"},
    ),
    Document(
        page_content="""Parental Leave Policy. Birthing parents are eligible for 12 weeks of fully paid parental leave. Non-birthing parents (including adoptive and foster parents) receive 8 weeks of fully paid leave. Leave must be taken within 12 months of the child's birth or adoption. Employees must notify HR at least 30 days before intended leave start date when foreseeable. During parental leave, health insurance benefits continue without interruption and the employee's position is protected.""",
        metadata={"source": "HR-002", "title": "Parental Leave", "department": "hr"},
    ),
    Document(
        page_content="""Remote Work Policy. Employees may work remotely up to 3 days per week with manager approval. Fully remote arrangements require VP-level approval and are evaluated case-by-case based on role requirements. Remote workers must maintain core working hours of 10 AM to 3 PM in their assigned time zone for meeting availability. The company provides a $500 one-time home office stipend for fully remote employees to purchase equipment. Hybrid employees receive $250.""",
        metadata={"source": "HR-003", "title": "Remote Work", "department": "hr"},
    ),
    Document(
        page_content="""Performance Review Process. Formal performance reviews occur twice per year: mid-year in June and annual in December. Reviews use a 5-point scale: 1 (Below Expectations), 2 (Partially Meets), 3 (Meets Expectations), 4 (Exceeds), 5 (Outstanding). Employees complete a self-assessment two weeks before the review meeting. Merit increases and promotion decisions are based on the annual December review. Employees rated 1 or 2 are placed on a 90-day Performance Improvement Plan (PIP).""",
        metadata={"source": "HR-004", "title": "Performance Reviews", "department": "hr"},
    ),
    Document(
        page_content="""Onboarding Process for New Hires. New employees complete a 30-day onboarding program. Week 1 focuses on orientation, IT setup, and benefits enrollment — employees must enroll in benefits within 30 days of start date. Week 2 covers department-specific training. Weeks 3-4 involve shadowing and initial project assignment. New hires are assigned an onboarding buddy and meet with their manager 1-on-1 weekly during onboarding. A 30-day check-in with HR is mandatory.""",
        metadata={"source": "HR-005", "title": "Onboarding", "department": "hr"},
    ),
    Document(
        page_content="""Code of Conduct and Anti-Harassment. The company maintains a zero-tolerance policy for harassment, discrimination, or retaliation of any kind. Reports can be made to any manager, HR Business Partner, or anonymously through the EthicsPoint hotline at 1-800-555-0199. All reports are investigated within 5 business days. Retaliation against anyone making a good-faith report is itself a terminable offense. Annual anti-harassment training is mandatory for all employees and must be completed by November 30 each year.""",
        metadata={"source": "HR-006", "title": "Code of Conduct", "department": "hr"},
    ),
    Document(
        page_content="""Learning and Development Benefits. Each employee receives an annual Learning & Development budget of $2,000 for approved courses, conferences, certifications, and books. Requests must be pre-approved by the employee's manager. The company also offers tuition reimbursement of up to $5,250 per year for degree programs directly related to the employee's role. Employees must remain with the company for 12 months after course completion or repay a prorated amount. Conference attendance requires VP approval if travel is involved.""",
        metadata={"source": "HR-007", "title": "Learning and Development", "department": "hr"},
    ),
    Document(
        page_content="""Health and Wellness Benefits. The company offers three medical plan tiers: Basic PPO, Premium PPO, and HDHP with HSA. Employee premiums are 20%, 15%, and 10% of total cost respectively. Dental and vision are included at no cost to the employee. The wellness program provides a $50/month gym membership reimbursement, free access to the Headspace meditation app, and an annual $300 wellness stipend for fitness equipment, wearables, or mental health services. Employees can use wellness funds for therapy copays.""",
        metadata={"source": "HR-008", "title": "Health and Wellness", "department": "hr"},
    ),
]

fin_docs = [
    Document(
        page_content="""Expense Reimbursement Policy. Employees can be reimbursed for pre-approved business expenses. All expenses must be submitted via Expensify within 30 days of being incurred. Receipts are required for any single expense over $25. Meals during business travel are reimbursable up to $75 per day for domestic travel and $125 per day for international travel. Alcohol is not reimbursable except at client entertainment events with manager approval. Personal expenses incorrectly charged to company cards must be repaid within 14 days.""",
        metadata={"source": "FIN-001", "title": "Expense Reimbursement", "department": "finance"},
    ),
    Document(
        page_content="""Corporate Travel Policy. Business travel requires manager approval in advance. Flights: economy class for flights under 6 hours; premium economy allowed for flights over 6 hours; business class requires VP approval. Hotels must not exceed $250/night in standard US cities, $350/night in NYC/SF/LA, $400/night internationally. Ground transportation: Uber, Lyft, and rental cars are reimbursable. Rental car class is limited to mid-size. Personal vehicle mileage is reimbursed at the current IRS rate of $0.67 per mile.""",
        metadata={"source": "FIN-002", "title": "Travel Policy", "department": "finance"},
    ),
    Document(
        page_content="""Budget Planning and Approval. The annual budget cycle begins October 1st. Department heads submit proposed budgets by October 31st. Finance reviews and consolidates by November 30th. Final budget approval by the CFO and CEO occurs in the December board meeting. Quarterly budget reviews occur in the first week of each new quarter. Any spending exceeding 110% of an approved line item requires CFO approval. Capital expenditures over $50,000 require board approval regardless of budget status.""",
        metadata={"source": "FIN-003", "title": "Budget Planning", "department": "finance"},
    ),
    Document(
        page_content="""Accounts Payable (AP) Process. Vendor invoices should be submitted to ap@company.com. Standard payment terms are Net 30 from invoice date. Early payment discounts of 2/10 Net 30 are negotiated with strategic vendors. All invoices over $10,000 require dual approval: the budget owner and a Finance manager. Invoices over $100,000 additionally require CFO approval. Vendors must be set up in Coupa with a completed W-9 (domestic) or W-8 (international) before their first invoice can be processed.""",
        metadata={"source": "FIN-004", "title": "Accounts Payable", "department": "finance"},
    ),
    Document(
        page_content="""Revenue Recognition Policy. The company follows ASC 606 for revenue recognition. Subscription revenue is recognized ratably over the contract term. Professional services revenue is recognized as services are delivered based on a percentage-of-completion method. One-time setup fees are deferred and recognized over the expected customer life of 3 years. Any contract modifications require review by the Revenue Operations team. Channel partner commissions are recognized at the point of contract signing, not collection.""",
        metadata={"source": "FIN-005", "title": "Revenue Recognition", "department": "finance"},
    ),
    Document(
        page_content="""Corporate Card Program. Employees in roles requiring frequent business expenses are eligible for a corporate American Express card. Card applications are approved by the employee's manager and Finance. Monthly reconciliation in Expensify is required by the 5th of the following month. Cards not reconciled within 60 days are automatically suspended. Cash advances are not permitted on corporate cards. Lost or stolen cards must be reported immediately to Amex (1-800-528-4800) and the corporate card administrator.""",
        metadata={"source": "FIN-006", "title": "Corporate Card", "department": "finance"},
    ),
    Document(
        page_content="""Financial Reporting and Close. The monthly close process runs from the 1st through the 5th business day of the following month. The Finance team publishes the preliminary P&L by day 6 and the board-ready financial package by day 10. Quarterly financial statements are prepared according to GAAP and reviewed by external auditors (KPMG). The fiscal year ends December 31. Annual audit typically completes in late February with the 10-K filed in March. SOX compliance controls apply to all financial processes.""",
        metadata={"source": "FIN-007", "title": "Financial Reporting", "department": "finance"},
    ),
    Document(
        page_content="""Procurement and Purchase Orders. All purchases over $5,000 require a formal purchase order (PO) created in Coupa before the order is placed. Vendors must be in good standing and properly set up. For purchases between $5,000 and $25,000, the budget owner and direct manager approve. Between $25,000 and $100,000, VP approval is required. Above $100,000, CFO approval is required. Sole-source justifications must accompany any PO that did not go through competitive bidding when value exceeds $50,000.""",
        metadata={"source": "FIN-008", "title": "Procurement", "department": "finance"},
    ),
]

print(f"HR: {len(hr_docs)}  |  Finance: {len(fin_docs)}")

HR: 8  |  Finance: 8


## 4. Neo4j Knowledge Graph — Hybrid Retrieval

In [5]:
from langchain_neo4j import Neo4jGraph, Neo4jVector
from langchain_text_splitters import RecursiveCharacterTextSplitter

graph = Neo4jGraph()
graph.query("MATCH (n) DETACH DELETE n")
print("Cleared existing graph.")

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
hr_chunks = splitter.split_documents(hr_docs)
fin_chunks = splitter.split_documents(fin_docs)
print(f"HR chunks: {len(hr_chunks)}  |  Finance chunks: {len(fin_chunks)}")

hr_vector = Neo4jVector.from_documents(
    documents=hr_chunks,
    embedding=embedder,
    index_name="hr_vector",
    node_label="HRDocument",
    text_node_property="text",
    embedding_node_property="embedding",
    search_type="hybrid",
    keyword_index_name="hr_keyword",
)

fin_vector = Neo4jVector.from_documents(
    documents=fin_chunks,
    embedding=embedder,
    index_name="fin_vector",
    node_label="FinDocument",
    text_node_property="text",
    embedding_node_property="embedding",
    search_type="hybrid",
    keyword_index_name="fin_keyword",
)

print("Indexes created.")
print(graph.query("MATCH (n) RETURN labels(n)[0] AS label, count(n) AS count"))

Cleared existing graph.
HR chunks: 12  |  Finance chunks: 13
Indexes created.
[{'label': 'HRDocument', 'count': 12}, {'label': 'FinDocument', 'count': 13}]


In [6]:
test_docs = hr_vector.similarity_search("How much PTO do I get?", k=2)
for i, d in enumerate(test_docs):
    print(f"[{i+1}] {d.metadata.get('title', '?')} — {d.page_content[:100]}...")
    print()

[1] PTO Policy — Paid Time Off (PTO) Policy. All full-time employees accrue 15 days of paid vacation per year during ...

[2] PTO Policy — 1st. PTO requests must be submitted through Workday at least two weeks in advance for requests longe...



## 5. Multi-Query Generation + Reciprocal Rank Fusion

In [7]:
import re
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.load import dumps, loads

MULTI_QUERY_TEMPLATE = """You are an AI language model assistant. Your task is to generate four
different versions of the given user question to retrieve relevant documents from a vector
database. Provide these alternative questions separated by newlines.

Original question: {question}

Note: Only return a list of questions under format:
1. question 1
2. question 2
3. question 3
4. question 4
without any explanation"""

multi_query_prompt = ChatPromptTemplate.from_template(MULTI_QUERY_TEMPLATE)

def parse_queries(text: str) -> list[str]:
    """Extract numbered list items from LLM-generated text.
    Args:
        text: Raw string output from an LLM, expected to contain a
            numbered list (e.g., "1. ...\n2. ...\n3. ...").

    Returns:
        A list of query strings with numbering and whitespace stripped.
        Returns an empty list if no numbered items are found.

    Example:
        >>> parse_queries("Here are queries:\\n1. What is RAG?\\n2. How does it work?")
        ['What is RAG?', 'How does it work?']
    """
    pattern = re.compile(r"^\s*\d+\.\s+(.+)$")
    return [m.group(1).strip() for line in text.split("\n") if (m := pattern.match(line))]

generate_queries = multi_query_prompt | llm | StrOutputParser() | parse_queries

# Test
sample = generate_queries.invoke({"question": "How much parental leave do I get?"})
for q in sample:
    print(f"  • {q}")

  • What is the amount of parental leave offered by my employer?
  • How many weeks of paid parental leave am I entitled to?
  • Can you tell me the duration of parental leave benefits in my country?
  • What are the rules for parental leave entitlements for new parents?


In [8]:
def reciprocal_rank_fusion(results: list[list], k: int = 60) -> list:
    """Combine multiple ranked document lists into a single re-ranked list using RRF.

    Reciprocal Rank Fusion scores each document by summing 1 / (rank + k)
    across all lists it appears in, rewarding documents that rank highly
    in multiple retrievals. Commonly used in multi-query RAG to merge
    results from several query reformulations into one final ranking.

    Args:
        results: A list of ranked document lists, where each inner list
            is ordered from most to least relevant (e.g., one list per
            query in multi-query retrieval).
        k: Smoothing constant that dampens the influence of top ranks.
            Defaults to 60, the value from the original RRF paper.

    Returns:
        A single list of documents sorted by fused score in descending
        order, with duplicates across input lists merged.
    """
    fused_scores = {}
    for docs in results:
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)
            fused_scores[doc_str] = fused_scores.get(doc_str, 0) + 1 / (rank + k)
    reranked = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    return [loads(doc_str) for doc_str, _ in reranked]

## 6. Cross-Encoder Reranker

In [9]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-large", max_length=512)

def rerank(query: str, docs: list, top_k: int = 4) -> list:
    """Re-rank retrieved documents against a query using a cross-encoder and return the top_k most relevant.

    Args:
        query: The user query string.
        docs: Candidate documents from initial retrieval (each with a .page_content attribute).
        top_k: Number of top documents to return. Defaults to 4.

    Returns:
        The top_k documents sorted by cross-encoder relevance score (descending).
        Empty list if docs is empty.

    Example:
        >>> rerank("What is LoRA?", retrieved_docs, top_k=3)
        [doc2, doc5, doc1]
    """
    if not docs:
        return []
    pairs = [(query, d.page_content) for d in docs]
    scores = reranker.predict(pairs)
    scored = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [d for d, _ in scored[:top_k]]

config.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

##7. HRAssistant + Finassistant

In [10]:
from langchain_core.runnables import RunnableLambda
from operator import itemgetter

RAG_TEMPLATE = """You are an assistant for the {department} department. Answer the user's
question based ONLY on the context below. If the context doesn't contain the answer,
say you don't have that information in your policies — do not guess.

Be concise. Cite the policy document title when relevant.

Context:
{context}

Question: {question}

Answer:"""

In [11]:
rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

In [12]:
def make_assistant(vector_store, department_name: str, retrieval_candidates: int = 10, final_k: int = 4):

    def retrieve_and_fuse(question: str) -> list:
        """
        this function takes user question as input and then pass into the generate queries
        to create 4 paraphrases of the same question.
        """
        alt_queries = generate_queries.invoke({"question": question})
        all_queries = [question] + alt_queries
        results = [vector_store.similarity_search(q, k=retrieval_candidates) for q in all_queries]
        fused = reciprocal_rank_fusion(results)
        return rerank(question, fused[:retrieval_candidates], top_k=final_k)

    def format_docs(docs: list) -> str:
        return "\n\n".join(f"[{d.metadata.get('title', '?')}] {d.page_content}" for d in docs)

    return (
        {
            "context": RunnableLambda(lambda x: x["question"]) | RunnableLambda(retrieve_and_fuse) | format_docs,
            "question": itemgetter("question"),
            "department": lambda _: department_name,
        }
        | rag_prompt
        | llm
        | StrOutputParser()
    )

hr_assistant = make_assistant(hr_vector, "HR")
fin_assistant = make_assistant(fin_vector, "Finance")
print("Assistants built.")

Assistants built.


In [13]:
print("HR ASSISTANT")
print("=" * 60)
print(hr_assistant.invoke({"question": "How much PTO do I accrue per year?"}))
print()
print(hr_assistant.invoke({"question": "What's the parental leave policy for non-birthing parents?"}))

HR ASSISTANT


/tmp/ipykernel_9927/2835526047.py:26: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc_str) for doc_str, _ in reranked]


You accrue 15 days of PTO per year during your first three years of employment. After three years, you accrue 20 days per year. (Source: PTO Policy)

Non-birthing parents (including adoptive and foster parents) are eligible for 8 weeks of fully paid parental leave, as stated in the Parental Leave Policy. Leave must be taken within 12 months of the child's birth or adoption.


In [14]:
print("FINANCE ASSISTANT")
print("=" * 60)
print(fin_assistant.invoke({"question": "What's the meal per diem for international travel?"}))
print()
print(fin_assistant.invoke({"question": "Do I need a PO for a $30,000 purchase?"}))

FINANCE ASSISTANT
The meal per diem for international travel is $125 per day. (Source: Expense Reimbursement Policy)

Yes, you need a purchase order (PO) for a $30,000 purchase. According to the [Procurement] Procurement and Purchase Orders policy, all purchases over $5,000 require a formal PO in Coupa. Since $30,000 falls between $25,000 and $100,000, VP approval is also required.


## 8. BossAssistant v1 — Logical Routing

In [15]:
from typing import Literal
from pydantic import BaseModel, Field

class RouteQuery(BaseModel):
    """Route a user question to the correct department assistant."""
    department: Literal["hr", "finance", "both"] = Field(
        ...,
        description=(
            "Which department should answer this question? "
            "'hr' for HR topics (PTO, benefits, remote work, performance reviews, onboarding, conduct). "
            "'finance' for Finance topics (expenses, travel reimbursement, budgets, AP, procurement, revenue). "
            "'both' if the question requires information from both departments."
        ),
    )

structured_llm = llm.with_structured_output(RouteQuery) # Wrap the LLM so it's constrained to return a valid RouteQuery object
# (via the provider's tool-calling API) instead of free-form text.

ROUTER_SYSTEM = """You are a router that decides which department's assistant should answer a user's question.

- HR covers: PTO, parental leave, remote work, performance reviews, onboarding, code of conduct, learning & development, health & wellness benefits.
- Finance covers: expense reimbursement, corporate travel costs, budgets, accounts payable, revenue recognition, corporate cards, financial reporting, procurement.

If a question requires information from BOTH departments (e.g., asking about the expense policy for an HR wellness benefit), return 'both'."""

router_prompt = ChatPromptTemplate.from_messages([
    ("system", ROUTER_SYSTEM),
    ("human", "{question}"),
])

logical_router = router_prompt | structured_llm

for q in [
    "How much PTO do I get?",
    "What's the hotel budget for business travel to NYC?",
    "Can I expense my Headspace subscription from my wellness stipend?",
]:
    result = logical_router.invoke({"question": q})
    print(f"  '{q[:50]}...'  →  {result.department}")

  'How much PTO do I get?...'  →  hr
  'What's the hotel budget for business travel to NYC...'  →  finance
  'Can I expense my Headspace subscription from my we...'  →  both
